# Interactive structures and trajectories in Jupyter: nglview and weas-widget

Install into your analysis environment (JupyterLab >= 4 recommended):

```bash
conda install -c conda-forge nglview ase      # or: pip install nglview ase
pip install weas-widget
```

Put this notebook next to the files you want to look at (POSCAR/CONTCAR, XDATCAR, *.xyz, *.cif)
or edit the file names below. The demo cells fall back to structures built with ASE, so the
notebook also runs without any DFT output. No output is stored in this file.

In [ ]:
import os
from ase.build import bulk, fcc111, add_adsorbate, molecule
from ase.io import read

def load_or_build(fname, builder):
    return read(fname) if os.path.isfile(fname) else builder()

# bulk GaAs (or your CONTCAR)
atoms = load_or_build("CONTCAR", lambda: bulk("GaAs", "zincblende", a=5.653))
# CO on Cu(111) as a slab example
def make_slab():
    slab = fcc111("Cu", size=(3, 3, 4), vacuum=10.0)
    add_adsorbate(slab, molecule("CO"), height=1.9, position="ontop")
    return slab
slab = make_slab()
print(atoms.get_chemical_formula(), len(slab), "atoms in the slab")

## 1. nglview: single structure with unit cell
Representations: `ball+stick`, `spacefill`, `licorice`, `line`. `add_unitcell()` draws the cell.

In [ ]:
import nglview as nv

v = nv.show_ase(atoms.repeat((2, 2, 2)))
v.clear_representations()
v.add_representation("ball+stick")
v.add_unitcell()
v.camera = "orthographic"
v

## 2. nglview: a trajectory (XDATCAR, OUTCAR, extxyz ...)
`read(..., index=":")` returns a list of `Atoms`; `show_asetraj` adds a frame slider.
For long AIMD runs read every n-th frame, e.g. `index="::10"`.

In [ ]:
if os.path.isfile("XDATCAR"):
    traj = read("XDATCAR", index="::10")
else:
    # demo: a short "trajectory" with growing random displacements
    import numpy as np
    traj = []
    for i in range(20):
        a = slab.copy()
        a.positions += np.random.default_rng(i).normal(scale=0.02 * i, size=a.positions.shape)
        traj.append(a)
tv = nv.show_asetraj(traj)
tv.add_unitcell()
tv

## 3. Saving a picture from nglview
`render_image()` renders in the browser; `download_image()` saves a PNG through the browser.
Run the cell after the widget above is displayed.

In [ ]:
tv.download_image(filename="slab_frame.png", factor=4, transparent=False)

## 4. weas-widget: viewer with editing, polyhedra, isosurfaces
`from_ase` accepts an `Atoms` object or a list of them (animation).
Model styles in the GUI: ball, ball+stick, polyhedra; atoms can be selected, moved and deleted,
and the edited structure is available again as `viewer.to_ase()`.

In [ ]:
from weas_widget import WeasWidget

viewer = WeasWidget()
viewer.from_ase(slab)
viewer

In [ ]:
# after editing in the widget: get the structure back and write it
edited = viewer.to_ase()
edited.write("POSCAR_edited", format="vasp", direct=True, sort=True)

## 5. Browser alternative without Python: matterviz
Drag a CIF/POSCAR/XDATCAR/CHGCAR onto https://matterviz.janosh.dev, or install the
MatterViz extension for VS Code / JupyterLab. Nothing has to be installed on the cluster.